In [ ]:
"""
Мини-анализатор логов веб-сервера.
Создан на основе объектно-ориентированного подхода.

Содержит три основных класса:
1) LogRecord представляет одну запись лога
2) LogParser разбирает строки с помощью регулярных выражений
3) LogAnalyzer загружает лог-файл и позволяет фильтровать и искать записи

Подходит для учебных целей (Питон, ООП, работа с файлами, regex).
"""

import re
from datetime import datetime


# =====================================================================
# 1. КЛАСС LOGRECORD — ОДНА СТРОКА ЛОГА
# =====================================================================
class LogRecord:
    """
    Класс, представляющий одну строку лог-файла.
    Хранит разобранные данные: дату, уровень, модуль и текст сообщения.
    """

    def __init__(self, timestamp, level, source, message, raw):
        # Сохраняем все поля — это удобно для анализа
        self.timestamp = timestamp   # строка с датой/временем
        self.level = level           # уровень логирования (INFO, ERROR…)
        self.source = source         # источник записи (auth, db, cache…)
        self.message = message       # содержимое сообщения
        self.raw = raw               # исходная строка целиком

    def __str__(self):
        """Возвращает красивое строковое представление записи."""
        return f"[{self.timestamp}] {self.level} ({self.source}) – {self.message}"

    # Дополнительные удобные методы
    def is_error(self):
        """Возвращает True, если запись является ошибкой."""
        return self.level.upper() == "ERROR"

    def is_warning(self):
        return self.level.upper() == "WARNING"

    def is_info(self):
        return self.level.upper() == "INFO"


# =====================================================================
# 2. КЛАСС LOGPARSER — РАЗБОР ОДНОЙ СТРОКИ ЧЕРЕЗ REGEX
# =====================================================================
class LogParser:
    """
    Парсер логов. Использует регулярное выражение с именованными группами:
    timestamp, level, source, message.
    """

    # Многострочные комментарии: здесь пример паттерна формата логов
    DEFAULT_PATTERN = re.compile(
        r"""
        ^(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2},\d{3})      # дата и время
        \s+
        (?P<level>[A-Z]+)                                               # уровень лога
        \s+
        \[(?P<source>[^\]]+)\]                                          # источник (в скобках)
        \s+
        (?P<message>.*)$                                                # сообщение
        """,
        re.VERBOSE  # позволяет использовать многострочные regex с комментариями
    )

    def __init__(self, pattern=None):
        # Если паттерн не передан — используем стандартный
        self.pattern = pattern or self.DEFAULT_PATTERN

    def parse_line(self, line: str):
        """
        Возвращает объект LogRecord, если строка подходит под паттерн.
        Если нет — возвращает None.
        """
        match = self.pattern.match(line)
        if not match:
            return None

        data = match.groupdict()  # словарь с timestamp, level, source, message

        return LogRecord(
            timestamp=data["timestamp"],
            level=data["level"],
            source=data["source"],
            message=data["message"],
            raw=line
        )


# =====================================================================
# 3. КЛАСС LOGANALYZER — ЗАГРУЗКА И АНАЛИЗ ЛОГОВ
# =====================================================================
class LogAnalyzer:
    """
    Отвечает за загрузку лог-файла, хранение записей и их анализ.
    """

    def __init__(self, filepath: str, parser: LogParser):
        self.filepath = filepath     # путь к файлу логов
        self.parser = parser         # объект LogParser
        self.records = []            # список LogRecord

    def load(self):
        """
        Загружает лог-файл построчно,
        каждую строку отправляет в парсер и сохраняет результат.
        """
        self.records.clear()  # очищаем на случай повторной загрузки
        with open(self.filepath, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()  # убираем перенос строки
                record = self.parser.parse_line(line)
                if record:
                    self.records.append(record)

    def filter_by_level(self, level: str):
        """Фильтрует записи по уровню (INFO, ERROR, WARNING…)."""
        return [r for r in self.records if r.level.upper() == level.upper()]

    def filter_by_source(self, source: str):
        """Фильтрует записи по источнику (auth, db, cache…)."""
        return [r for r in self.records if r.source.lower() == source.lower()]

    def search_in_message(self, pattern: str):
        """
        Ищет совпадения в тексте сообщения по регулярному выражению.
        Пример: "timeout" или r"User \"(\w+)\""
        """
        reg = re.compile(pattern, re.IGNORECASE)
        return [r for r in self.records if reg.search(r.message)]

    def get_stats_by_level(self):
        """Возвращает количество записей каждого уровня в виде словаря."""
        stats = {}
        for r in self.records:
            stats[r.level] = stats.get(r.level, 0) + 1
        return stats


# =====================================================================
# БЛОК ТЕСТИРОВАНИЯ
# Этот код выполняется только если запускать файл напрямую
# =====================================================================
if name == "__main__":
    """
    Для запуска теста в VS Code:
    1) Создать файл log.txt с логами
    2) Запустить этот скрипт
    """

    parser = LogParser()
    analyzer = LogAnalyzer("log.txt", parser)

    analyzer.load()

    print("\nВсе записи:")
    for r in analyzer.records:
        print(r)

    print("\nОшибки:")
    for r in analyzer.filter_by_level("ERROR"):
        print(r)

    print("\nПоиск слова 'timeout':")
    for r in analyzer.search_in_message("timeout"):
        print(r)

    print("\nСтатистика по уровням:")
    print(analyzer.get_stats_by_level())